In [1]:
import pandas as pd
import numpy as np

In [2]:
op_price_data = pd.read_csv("../../../data/votable_supply-dhruvi/data/lar_raw_data.csv")
avpi_data = pd.read_csv("../../../data/votable_supply-dhruvi/data/avpi.csv")
pr_data = pd.read_csv("../../../data/votable_supply-dhruvi/data/pr.csv")
lar_data = pd.read_csv("../../../data/votable_supply-dhruvi/data/lar.csv")
psi_data = pd.read_csv("../../../data/votable_supply-dhruvi/data/psi.csv")
vpi_data = pd.read_csv("../../../data/votable_supply-dhruvi/data/vpi.csv")

In [3]:
print("OP Price Data Columns:")
print(op_price_data.columns)

print(f"\nAVPI Data Columns:")
print(avpi_data.columns)

print(f"\nPR Data Columns:")
print(pr_data.columns)

print(f"\nLAR Data Columns:")
print(lar_data.columns)

print(f"\nPSI Data Columns:")
print(psi_data.columns)

print(f"\nVPI Data Columns:")
print(vpi_data.columns)

OP Price Data Columns:
Index(['Date', 'Price (USD)', 'Market Cap (USD)', 'Total Volume (USD)',
       'Circulating Supply'],
      dtype='object')

AVPI Data Columns:
Index(['Date', 'Total Volume (USD)', 'Actual Votable Supply ',
       '30 Day Avg Total Volume (USD)', 'AVPI'],
      dtype='object')

PR Data Columns:
Index(['Date', 'Votable Supply', 'Circulating Supply', 'PR'], dtype='object')

LAR Data Columns:
Index(['Date', 'Price (USD)', 'Market Cap (USD)', 'Total Volume (USD)',
       'Circulating Supply', 'LAR'],
      dtype='object')

PSI Data Columns:
Index(['Date', 'Price (USD)', 'Market Cap (USD)', 'Total Volume (USD)',
       'Circulating Supply', '7-day Avg Price', 'PSI'],
      dtype='object')

VPI Data Columns:
Index(['Date', 'Votable Supply', 'Total Volume (USD)',
       '30 Day Avg Total Volume (USD)', 'VPI'],
      dtype='object')


### Drop the Columns That Are Not Required

In [4]:
op_price_data = op_price_data.drop(['Market Cap (USD)', 'Total Volume (USD)', 'Circulating Supply'], axis=1)
avpi_data = avpi_data.drop(['Total Volume (USD)', 'Actual Votable Supply ', '30 Day Avg Total Volume (USD)'], axis=1)
pr_data = pr_data.drop(['Votable Supply', 'Circulating Supply'], axis=1)
lar_data = lar_data.drop(['Price (USD)', 'Market Cap (USD)', 'Total Volume (USD)', 'Circulating Supply'], axis=1)
psi_data = psi_data.drop(['Price (USD)', 'Market Cap (USD)', 'Total Volume (USD)', 'Circulating Supply', '7-day Avg Price'], axis=1)
vpi_data = vpi_data.drop(['Votable Supply', 'Total Volume (USD)', '30 Day Avg Total Volume (USD)'], axis=1)

In [5]:
print(op_price_data.columns)
print(avpi_data.columns)
print(pr_data.columns)
print(lar_data.columns)
print(psi_data.columns)
print(vpi_data.columns)

Index(['Date', 'Price (USD)'], dtype='object')
Index(['Date', 'AVPI'], dtype='object')
Index(['Date', 'PR'], dtype='object')
Index(['Date', 'LAR'], dtype='object')
Index(['Date', 'PSI'], dtype='object')
Index(['Date', 'VPI'], dtype='object')


### Rename the Columns

In [6]:
# Rename the column 'Price (USD)' to 'OP_Price'
op_price_data = op_price_data.rename(columns={'Price (USD)': 'OP_Price'})

### Scale OP Price Data

In [7]:
# Sigmoid scaling of OP_Price
k = 1.5  
x0 = op_price_data['OP_Price'].mean() 
op_price_data['OP_Price_Scaled'] = 1 / (1 + np.exp(-k * (op_price_data['OP_Price'] - x0)))

### Scale LAR Data

In [8]:
# Sigmoid scaling of Liquidity Activity Ratio (LAR)
k = 5  
x0 = lar_data['LAR'].mean() 
lar_data['LAR_Scaled'] = 1 / (1 + np.exp(-k * (lar_data['LAR'] - x0)))

### Scale PSI Data

In [9]:
# Sigmoid scaling of Price Stability Index (PSI)
k = 8
x0 = psi_data['PSI'].mean() 
psi_data['PSI_Scaled'] = 1 / (1 + np.exp(-k * (psi_data['PSI'] - x0)))

### Merge all Dataframes

In [10]:
op_price_data['Date'] = pd.to_datetime(op_price_data['Date'], dayfirst=True)
avpi_data['Date'] = pd.to_datetime(avpi_data['Date'], dayfirst=True)
pr_data['Date'] = pd.to_datetime(pr_data['Date'], dayfirst=True)
lar_data['Date'] = pd.to_datetime(lar_data['Date'], dayfirst=True)
psi_data['Date'] = pd.to_datetime(psi_data['Date'], dayfirst=True)
vpi_data['Date'] = pd.to_datetime(vpi_data['Date'], dayfirst=True)

In [11]:
merged_data = op_price_data.merge(avpi_data, on='Date', how='inner')\
    .merge(pr_data, on='Date', how='inner')\
    .merge(lar_data, on='Date', how='inner')\
    .merge(psi_data, on='Date', how='inner')\
    .merge(vpi_data, on='Date', how='inner')

In [12]:
merged_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 363 entries, 0 to 362
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             363 non-null    datetime64[ns]
 1   OP_Price         363 non-null    float64       
 2   OP_Price_Scaled  363 non-null    float64       
 3   AVPI             363 non-null    float64       
 4   PR               363 non-null    float64       
 5   LAR              363 non-null    float64       
 6   LAR_Scaled       363 non-null    float64       
 7   PSI              363 non-null    float64       
 8   PSI_Scaled       363 non-null    float64       
 9   VPI              363 non-null    float64       
dtypes: datetime64[ns](1), float64(9)
memory usage: 28.5 KB


### Merge the Votable Supply and Circulating Supply

In [13]:
vs_cs_data = pd.read_csv("../../../data/votable_supply-dhruvi/data/pr_raw_data.csv")

In [14]:
vs_cs_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 910 entries, 0 to 909
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                910 non-null    object 
 1   Votable Supply      910 non-null    float64
 2   Circulating Supply  910 non-null    float64
dtypes: float64(2), object(1)
memory usage: 21.5+ KB


In [15]:
vs_cs_data['Date'] = pd.to_datetime(vs_cs_data['Date'], dayfirst=True)

index_data = pd.merge(merged_data, vs_cs_data, on='Date')

In [16]:
index_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 363 entries, 0 to 362
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Date                363 non-null    datetime64[ns]
 1   OP_Price            363 non-null    float64       
 2   OP_Price_Scaled     363 non-null    float64       
 3   AVPI                363 non-null    float64       
 4   PR                  363 non-null    float64       
 5   LAR                 363 non-null    float64       
 6   LAR_Scaled          363 non-null    float64       
 7   PSI                 363 non-null    float64       
 8   PSI_Scaled          363 non-null    float64       
 9   VPI                 363 non-null    float64       
 10  Votable Supply      363 non-null    float64       
 11  Circulating Supply  363 non-null    float64       
dtypes: datetime64[ns](1), float64(11)
memory usage: 34.2 KB


### Round the decimals

In [17]:
index_data['OP_Price'] = round(index_data['OP_Price'], 6)
index_data['OP_Price_Scaled'] = round(index_data['OP_Price_Scaled'], 6)
index_data['AVPI'] = round(index_data['AVPI'], 6)
index_data['PR'] = round(index_data['PR'], 6)
index_data['LAR'] = round(index_data['LAR'], 6)
index_data['LAR_Scaled'] = round(index_data['LAR_Scaled'], 6)
index_data['PSI'] =  round(index_data['PSI'], 6)
index_data['PSI_Scaled'] =  round(index_data['PSI_Scaled'], 6)
index_data['VPI'] =  round(index_data['VPI'], 6)
index_data['Votable Supply'] = round(index_data['Votable Supply'], 2)
index_data['Circulating Supply'] = round(index_data['Circulating Supply'], 2)

### Save the Data to CSV

In [18]:
index_data.to_csv("../../../data/votable_supply-dhruvi/data/indexes.csv", index=False)